# Init

## Function

In [156]:
import torch
import os
from diffusers import (
    AutoencoderKL,
    DDPMScheduler,
    DiffusionPipeline,
    DPMSolverMultistepScheduler,
    DDIMScheduler,
    StableDiffusionPipeline,
    UNet2DConditionModel,
)
from safetensors.torch import load_file
import torch.nn.functional as F
import torch
import pandas as pd
from torch.nn.functional import cosine_similarity
from tqdm import tqdm
from copy import deepcopy
device = 'cuda:3'

In [157]:
def flatten_list(x):
    out = []
    for item in x:
        if isinstance(item, list):
            out.extend(flatten_list(item))
        else:
            out.append(item)
    return out


In [158]:
def build_unet_features(text_embeddings,unet):
    
    p = text_embeddings
    unet_params = dict(unet.named_parameters())
    
    unet_features = {}
    for name, W in unet_params.items():
        if 'attn2' not in name: continue
        # if 'mid' not in name: continue
        # if 'to_k' not in name: continue
        if not (name.endswith("to_k.weight") or name.endswith("to_v.weight")): continue
        
        # get bias if exists
        b_name = name.replace(".weight", ".bias")
        b = unet_params.get(b_name, None)
        if b is not None:b = b.detach()
                
                
        with torch.no_grad():
            W_p = F.linear(p, W, b)
            
        unet_features[name] = W_p
        
    return unet_features
            
def build_features(prompt, feature_option="text", is_normalize=True, return_eos=True, unet=None):
    global pipe, device

    prompts = [prompt] if isinstance(prompt, str) else list(prompt)
    print("num prompts:", len(prompts))

    tokenized = pipe.tokenizer(
        prompts,
        padding="max_length",
        truncation=True,
        max_length=pipe.tokenizer.model_max_length,
        return_tensors="pt",
    )

    token_ids = tokenized.input_ids  # [B, 77]

    tokens = [
        pipe.tokenizer.convert_ids_to_tokens(ids.tolist())
        for ids in token_ids
    ]
    print(tokens)
    eos_token_id = pipe.tokenizer.eos_token_id

    eos_indices = torch.tensor(
        [ids.tolist().index(eos_token_id) for ids in token_ids],
        device=device,
        dtype=torch.long,
    )  # [B]

    eos_tokens = [
        tokens[i][eos_indices[i].item()]
        for i in range(len(prompts))
    ]

    print(eos_indices)

    with torch.no_grad():
        text_embeddings = pipe.encode_prompt(
            prompt=prompts,
            device=device,
            num_images_per_prompt=1,
            do_classifier_free_guidance=False,
        )[0]  # [B, 77, D]

    if feature_option == "text":
        if is_normalize:
            text_embeddings = F.normalize(text_embeddings, p=2, dim=-1)

        if return_eos:
            batch_idx = torch.arange(text_embeddings.shape[0], device=device)
            text_embeddings = text_embeddings[batch_idx, eos_indices]
            # [B, D]

            return text_embeddings, eos_tokens

        return text_embeddings, tokens

    elif feature_option == "unet":
        all_unet_features = {}

        for i in range(len(prompts)):
            single_text_embedding = text_embeddings[i]  # [77, D]

            unet_features = build_unet_features(
                single_text_embedding,
                unet if unet is not None else pipe.unet,
            )
            # dict:
            # layer_name -> [77, D_layer]

            if i == 0:
                print(unet_features.keys())

            for layer_name, feature in unet_features.items():
                if is_normalize:
                    feature = F.normalize(feature, p=2, dim=-1)

                if return_eos:
                    feature = feature[eos_indices[i]]
                    # [D_layer]

                if layer_name not in all_unet_features:
                    all_unet_features[layer_name] = []

                all_unet_features[layer_name].append(feature)

        # Stack each layer independently
        for layer_name in all_unet_features:
            all_unet_features[layer_name] = torch.stack(
                all_unet_features[layer_name],
                dim=0,
            )

            # if return_eos=True:
            #   [B, D_layer]
            #
            # if return_eos=False:
            #   [B, 77, D_layer]

        if return_eos:
            return all_unet_features, eos_tokens

        return all_unet_features, tokens

    else:
        raise ValueError(f"Unknown feature_option: {feature_option}")

In [159]:
import torch
import pandas as pd
import torch.nn.functional as F
from tqdm.auto import tqdm


def compute_stats(x, eps=1e-8):
    """
    Compute basic statistics.

    CV = coefficient of variation = std / |mean|
    """
    x = x.float().flatten()

    mean = x.mean()
    std = x.std(unbiased=False)

    return {
        "mean": mean.item(),
        "average": mean.item(),
        "std": std.item(),
        "cv": (std / (mean.abs() + eps)).item(),
        "cv_percent": (100 * std / (mean.abs() + eps)).item(),
        "min": x.min().item(),
        "max": x.max().item(),
    }


def _ensure_feature_dict(features, feature_option):
    """
    Make output compatible with both UNet and text features.

    UNet features:
        dict[layer_name] -> tensor

    Text features:
        tensor -> {"text": tensor}
    """
    if isinstance(features, dict):
        return features

    return {feature_option: features}


def _paired_cosine(x, y):
    """
    Compute cosine similarity for paired prompts.

    Compare:
        x[i] with y[i]

    Input:
        x: [N, D] or [N, ..., D]
        y: [N, D] or [N, ..., D]

    Output:
        cos_sim: [N]
    """
    assert x.shape == y.shape, (
        f"Shape mismatch for paired cosine: x={x.shape}, y={y.shape}"
    )

    cos_sim = F.cosine_similarity(x, y, dim=-1)

    # If there are extra spatial/token dimensions, average them.
    # Example:
    # [N, H, W] -> [N]
    # [N, num_tokens] -> [N]
    if cos_sim.ndim > 1:
        cos_sim = cos_sim.mean(dim=tuple(range(1, cos_sim.ndim)))

    assert cos_sim.ndim == 1, (
        f"Unexpected paired cosine shape: {cos_sim.shape}"
    )

    return cos_sim


def _all_pair_cosine(x, y):
    """
    Compute cosine similarity for all target-generic combinations.

    Compare:
        every x[i] with every y[j]

    Input:
        x: [N_target, D] or [N_target, ..., D]
        y: [N_generic, D] or [N_generic, ..., D]

    Output:
        cos_sim: [N_target, N_generic]
    """
    assert x.shape[1:] == y.shape[1:], (
        f"Feature dimensions must match except batch dimension: "
        f"x={x.shape}, y={y.shape}"
    )

    cos_sim = F.cosine_similarity(
        x[:, None, ...],
        y[None, :, ...],
        dim=-1,
    )

    # If there are extra spatial/token dimensions, average them.
    # Example:
    # [N_target, N_generic, H, W] -> [N_target, N_generic]
    # [N_target, N_generic, num_tokens] -> [N_target, N_generic]
    if cos_sim.ndim > 2:
        cos_sim = cos_sim.mean(dim=tuple(range(2, cos_sim.ndim)))

    assert cos_sim.ndim == 2, (
        f"Unexpected all-pair cosine shape: {cos_sim.shape}"
    )

    return cos_sim


def _display_or_print(df):
    """
    Display DataFrame nicely in notebook.
    Fall back to print if display is unavailable.
    """
    try:
        display(df)
    except NameError:
        print(df)


def summarize_target_generic_similarity(
    target_prompts,
    generic_prompts,
    feature_option="unet",
    distance_aggregate_option="same_pair",  # "same_pair" or "all_pair"
    exclude_samepair=False,
    show_negative_pair=False,
    max_negative_pairs=50,
    is_normalize=True,
    return_eos=True,
    verbose=True,
    make_pairwise_df=False,
    print_only_summary=True,
):
    """
    Compute target-generic cosine similarity statistics.

    Parameters
    ----------
    target_prompts : list[str] or str
        Target / erased concept prompts.

    generic_prompts : list[str] or str
        Generic / anchor prompts.

    feature_option : str
        Feature type passed into build_features.
        Usually "unet" or "text".

    distance_aggregate_option : str
        "same_pair":
            Compare target_prompts[i] with generic_prompts[i].
            Output shape:
                [num_layers, num_pairs]

        "all_pair":
            Compare every target prompt with every generic prompt.
            Output shape before exclusion:
                [num_layers, num_targets, num_generics]

    exclude_samepair : bool
        Only used when distance_aggregate_option="all_pair".

        If True, remove diagonal pairs:
            target_prompts[i] <-> generic_prompts[i]

    show_negative_pair : bool
        If True, show all negative layer × concept-pair cosine values.

    max_negative_pairs : int or None
        Maximum number of negative entries to show.
        If None, show all negative entries.

    make_pairwise_df : bool
        Whether to create per-pair DataFrame.
        For ImageNet-1000 all-pair, keep this False.

    Returns
    -------
    results : dict
        Contains tensors, summaries, optional DataFrames,
        and negative-pair diagnostics.
    """

    assert distance_aggregate_option in ["same_pair", "all_pair"], (
        "distance_aggregate_option must be either 'same_pair' or 'all_pair'"
    )

    if isinstance(target_prompts, str):
        target_prompts = [target_prompts]

    if isinstance(generic_prompts, str):
        generic_prompts = [generic_prompts]

    if distance_aggregate_option == "same_pair":
        assert len(target_prompts) == len(generic_prompts), (
            f"For same_pair, target_prompts and generic_prompts must have "
            f"the same length. Got {len(target_prompts)} and "
            f"{len(generic_prompts)}."
        )

        if exclude_samepair:
            print(
                "Warning: exclude_samepair=True is ignored when "
                "distance_aggregate_option='same_pair'."
            )

    if distance_aggregate_option == "all_pair" and exclude_samepair:
        assert len(target_prompts) == len(generic_prompts), (
            "exclude_samepair=True requires target_prompts and generic_prompts "
            "to have the same length."
        )

    # ==================================================
    # 1. Build features
    # ==================================================

    p_e, _ = build_features(
        target_prompts,
        feature_option=feature_option,
        is_normalize=is_normalize,
        return_eos=return_eos,
    )

    p_g, _ = build_features(
        generic_prompts,
        feature_option=feature_option,
        is_normalize=is_normalize,
        return_eos=return_eos,
    )

    p_e = _ensure_feature_dict(p_e, feature_option)
    p_g = _ensure_feature_dict(p_g, feature_option)

    assert set(p_e.keys()) == set(p_g.keys()), "Layer keys do not match"

    layer_names = list(p_e.keys())

    # ==================================================
    # 2. Compute cosine similarity for each layer
    # ==================================================

    cos_sim_by_layer = {}

    for layer_name in layer_names:
        x = p_e[layer_name]
        y = p_g[layer_name]

        if distance_aggregate_option == "same_pair":
            cos_sim = _paired_cosine(x, y)
            # [num_pairs]

        elif distance_aggregate_option == "all_pair":
            cos_sim = _all_pair_cosine(x, y)
            # [num_targets, num_generics]

        cos_sim_by_layer[layer_name] = cos_sim

    # ==================================================
    # 3. Stack cosine similarities
    # ==================================================

    cos_tensor = torch.stack(
        [cos_sim_by_layer[layer_name] for layer_name in layer_names],
        dim=0,
    )

    num_layers = cos_tensor.shape[0]

    # ==================================================
    # 4. Flatten valid pairs
    # ==================================================

    pair_names = None
    valid_pair_indices = None

    if distance_aggregate_option == "same_pair":
        cos_by_layer_and_pair = cos_tensor.reshape(num_layers, -1)
        # [num_layers, num_pairs]

        num_pairs = len(target_prompts)

        valid_pair_indices = torch.stack(
            [
                torch.arange(num_pairs, device=cos_tensor.device),
                torch.arange(num_pairs, device=cos_tensor.device),
            ],
            dim=1,
        )
        # [num_pairs, 2], where each row = [target_idx, generic_idx]

        if make_pairwise_df:
            pair_names = [
                f"{t}  <->  {g}"
                for t, g in zip(target_prompts, generic_prompts)
            ]

    elif distance_aggregate_option == "all_pair":
        num_targets = len(target_prompts)
        num_generics = len(generic_prompts)

        assert cos_tensor.shape[1] == num_targets, (
            f"Expected num_targets={num_targets}, got {cos_tensor.shape[1]}"
        )
        assert cos_tensor.shape[2] == num_generics, (
            f"Expected num_generics={num_generics}, got {cos_tensor.shape[2]}"
        )

        pair_mask = torch.ones(
            num_targets,
            num_generics,
            dtype=torch.bool,
            device=cos_tensor.device,
        )

        if exclude_samepair:
            diagonal_idx = torch.arange(num_targets, device=cos_tensor.device)
            pair_mask[diagonal_idx, diagonal_idx] = False

        # valid_pair_indices[k] = [target_idx, generic_idx]
        valid_pair_indices = pair_mask.nonzero(as_tuple=False)

        # cos_tensor: [num_layers, num_targets, num_generics]
        # pair_mask:  [num_targets, num_generics]
        # output:     [num_layers, num_valid_pairs]
        cos_by_layer_and_pair = cos_tensor[:, pair_mask]

        if make_pairwise_df:
            pair_names = [
                f"{target_prompts[i]}  <->  {generic_prompts[j]}"
                for i, j in valid_pair_indices.detach().cpu().tolist()
            ]

    num_valid_pairs = cos_by_layer_and_pair.shape[1]

    if make_pairwise_df:
        assert pair_names is not None
        assert len(pair_names) == num_valid_pairs, (
            f"Pair-name mismatch: {len(pair_names)} names, "
            f"but tensor has {num_valid_pairs} valid pairs."
        )

    # ==================================================
    # 5. Negative pair diagnostics
    # ==================================================

    negative_pairs_df = None
    negative_summary = {
        "num_negative_values": 0,
        "negative_fraction_percent": 0.0,
        "most_negative_value": None,
    }

    if show_negative_pair:
        negative_mask = cos_by_layer_and_pair < 0
        num_negative_values = int(negative_mask.sum().item())
        total_values = int(negative_mask.numel())

        negative_summary["num_negative_values"] = num_negative_values
        negative_summary["negative_fraction_percent"] = (
            100.0 * num_negative_values / max(total_values, 1)
        )

        if num_negative_values > 0:
            neg_layer_idx, neg_pair_idx = negative_mask.nonzero(as_tuple=True)
            neg_values = cos_by_layer_and_pair[neg_layer_idx, neg_pair_idx]

            # Sort from most negative to less negative.
            order = torch.argsort(neg_values)

            if max_negative_pairs is not None:
                order = order[:max_negative_pairs]

            negative_summary["most_negative_value"] = neg_values.min().item()

            rows = []

            for rank, k in enumerate(order.detach().cpu().tolist(), start=1):
                layer_idx = int(neg_layer_idx[k].item())
                pair_idx = int(neg_pair_idx[k].item())

                target_idx = int(valid_pair_indices[pair_idx, 0].item())
                generic_idx = int(valid_pair_indices[pair_idx, 1].item())

                rows.append(
                    {
                        "rank": rank,
                        "cosine": cos_by_layer_and_pair[
                            layer_idx, pair_idx
                        ].item(),
                        "layer_idx": layer_idx,
                        "layer_name": layer_names[layer_idx],
                        "pair_idx": pair_idx,
                        "target_idx": target_idx,
                        "generic_idx": generic_idx,
                        "target_prompt": target_prompts[target_idx],
                        "generic_prompt": generic_prompts[generic_idx],
                    }
                )

            negative_pairs_df = pd.DataFrame(rows)

    # ==================================================
    # 6. All pooled stats
    # ==================================================

    all_stats = compute_stats(cos_by_layer_and_pair)

    # ==================================================
    # 7. Across layers: per concept-pair stats
    # ==================================================

    if make_pairwise_df:
        across_layers_stats = {}

        for pair_idx, pair_name in tqdm(
            enumerate(pair_names),
            total=len(pair_names),
            desc="Computing per-pair stats",
        ):
            values_across_layers = cos_by_layer_and_pair[:, pair_idx]
            across_layers_stats[pair_name] = compute_stats(values_across_layers)

        across_layers_df = pd.DataFrame(across_layers_stats).T

    else:
        across_layers_df = None

    pair_mean_over_layers = cos_by_layer_and_pair.mean(dim=0)
    pair_std_over_layers = cos_by_layer_and_pair.std(dim=0, unbiased=False)

    across_layers_summary = {
        "summary_of_pair_means_over_layers": compute_stats(pair_mean_over_layers),
        "summary_of_pair_stds_over_layers": compute_stats(pair_std_over_layers),
    }

    # ==================================================
    # 8. Across concept pairs: per-layer stats
    # ==================================================

    across_pairs_stats = {}

    for layer_idx, layer_name in enumerate(layer_names):
        values_across_pairs = cos_by_layer_and_pair[layer_idx, :]
        across_pairs_stats[layer_name] = compute_stats(values_across_pairs)

    across_pairs_df = pd.DataFrame(across_pairs_stats).T

    layer_mean_over_pairs = cos_by_layer_and_pair.mean(dim=1)
    layer_std_over_pairs = cos_by_layer_and_pair.std(dim=1, unbiased=False)

    across_pairs_summary = {
        "summary_of_layer_means_over_pairs": compute_stats(layer_mean_over_pairs),
        "summary_of_layer_stds_over_pairs": compute_stats(layer_std_over_pairs),
    }

    # ==================================================
    # 9. Intuitive final summary
    # ==================================================

    exclusion_note = (
        " Same-index target-generic pairs were excluded."
        if (distance_aggregate_option == "all_pair" and exclude_samepair)
        else ""
    )

    concise_summary = {
        "overall similarity": {
            "comment": (
                "All layers × all valid concept pairs pooled together."
                + exclusion_note
            ),
            "mean": all_stats["mean"],
            "std": all_stats["std"],
            "cv_percent": all_stats["cv_percent"],
            "min": all_stats["min"],
            "max": all_stats["max"],
        },

        "concept-wise inconsistency": {
            "comment": (
                "Variation across concept pairs after averaging over layers. "
                "This is the main concept-wise inconsistency score."
                + exclusion_note
            ),
            "mean": across_layers_summary[
                "summary_of_pair_means_over_layers"
            ]["mean"],
            "std": across_layers_summary[
                "summary_of_pair_means_over_layers"
            ]["std"],
            "cv_percent": across_layers_summary[
                "summary_of_pair_means_over_layers"
            ]["cv_percent"],
            "min": across_layers_summary[
                "summary_of_pair_means_over_layers"
            ]["min"],
            "max": across_layers_summary[
                "summary_of_pair_means_over_layers"
            ]["max"],
        },

        "layer-wise inconsistency": {
            "comment": (
                "Variation across layers after averaging over valid concept pairs. "
                "This is the main layer-wise inconsistency score."
                + exclusion_note
            ),
            "mean": across_pairs_summary[
                "summary_of_layer_means_over_pairs"
            ]["mean"],
            "std": across_pairs_summary[
                "summary_of_layer_means_over_pairs"
            ]["std"],
            "cv_percent": across_pairs_summary[
                "summary_of_layer_means_over_pairs"
            ]["cv_percent"],
            "min": across_pairs_summary[
                "summary_of_layer_means_over_pairs"
            ]["min"],
            "max": across_pairs_summary[
                "summary_of_layer_means_over_pairs"
            ]["max"],
        },

        "layer variation within each concept pair": {
            "comment": (
                "For each fixed valid concept pair, measure how much similarity "
                "changes across layers; then summarize over concept pairs."
                + exclusion_note
            ),
            "mean": across_layers_summary[
                "summary_of_pair_stds_over_layers"
            ]["mean"],
            "std": across_layers_summary[
                "summary_of_pair_stds_over_layers"
            ]["std"],
            "cv_percent": across_layers_summary[
                "summary_of_pair_stds_over_layers"
            ]["cv_percent"],
            "min": across_layers_summary[
                "summary_of_pair_stds_over_layers"
            ]["min"],
            "max": across_layers_summary[
                "summary_of_pair_stds_over_layers"
            ]["max"],
        },

        "concept variation within each layer": {
            "comment": (
                "For each fixed layer, measure how much similarity changes "
                "across valid concept pairs; then summarize over layers."
                + exclusion_note
            ),
            "mean": across_pairs_summary[
                "summary_of_layer_stds_over_pairs"
            ]["mean"],
            "std": across_pairs_summary[
                "summary_of_layer_stds_over_pairs"
            ]["std"],
            "cv_percent": across_pairs_summary[
                "summary_of_layer_stds_over_pairs"
            ]["cv_percent"],
            "min": across_pairs_summary[
                "summary_of_layer_stds_over_pairs"
            ]["min"],
            "max": across_pairs_summary[
                "summary_of_layer_stds_over_pairs"
            ]["max"],
        },
    }

    # ==================================================
    # 10. Collect results
    # ==================================================

    results = {
        "cos_tensor": cos_tensor,
        "cos_by_layer_and_pair": cos_by_layer_and_pair,
        "cos_sim_by_layer": cos_sim_by_layer,

        "layer_names": layer_names,

        "pair_names": pair_names,
        "valid_pair_indices": valid_pair_indices,
        "num_valid_pairs": num_valid_pairs,
        "exclude_samepair": exclude_samepair,

        "negative_summary": negative_summary,
        "negative_pairs_df": negative_pairs_df,

        "all_stats": all_stats,

        "across_layers_df": across_layers_df,
        "across_layers_summary": across_layers_summary,

        "across_pairs_df": across_pairs_df,
        "across_pairs_summary": across_pairs_summary,

        "concise_summary": concise_summary,
    }

    # ==================================================
    # 11. Print report
    # ==================================================

    if verbose:


        if print_only_summary:
           for key, value in concise_summary.items():
                print(f"\n{key}")
                print("-" * len(key))
                print(f"comment: {value['comment']}")
                print(
                    f"mean={value['mean']:.4f}, "
                    f"std={value['std']:.4f}, "
                    f"CV={value['cv_percent']:.2f}%, "
                    f"min={value['min']:.4f}, "
                    f"max={value['max']:.4f}"
                )
        else: 
            print("\n==============================")
            print("Cosine tensor shape")
            print("==============================")
            print(cos_tensor.shape)

            print("\n==============================")
            print("Flattened valid cosine shape")
            print("==============================")
            print(cos_by_layer_and_pair.shape)
            print(f"num_valid_pairs = {num_valid_pairs}")
            print(f"exclude_samepair = {exclude_samepair}")

            print("\n==============================")
            print("1. All pooled stats")
            print("==============================")
            print(all_stats)

            if show_negative_pair:
                print("\n==============================")
                print("Negative cosine diagnostics")
                print("==============================")
                print(negative_summary)

                if negative_pairs_df is not None and len(negative_pairs_df) > 0:
                    _display_or_print(negative_pairs_df)
                else:
                    print("No negative cosine values found.")

            if make_pairwise_df:
                print("\n==============================")
                print("2. Across layers: per concept-pair stats")
                print("==============================")
                _display_or_print(across_layers_df)
            else:
                print("\n==============================")
                print("2. Across layers: per concept-pair stats")
                print("==============================")
                print("Skipped per-pair DataFrame because make_pairwise_df=False.")

            print("\n==============================")
            print("2b. Across-layers summary")
            print("==============================")
            print(across_layers_summary)

            print("\n==============================")
            print("3. Across concept pairs: per-layer stats")
            print("==============================")
            _display_or_print(across_pairs_df)

            print("\n==============================")
            print("3b. Across-concept-pairs summary")
            print("==============================")
            print(across_pairs_summary)

            print("\n==============================")
            print("4. Intuitive final summary")
            print("==============================")

            for key, value in concise_summary.items():
                print(f"\n{key}")
                print("-" * len(key))
                print(f"comment: {value['comment']}")
                print(
                    f"mean={value['mean']:.4f}, "
                    f"std={value['std']:.4f}, "
                    f"CV={value['cv_percent']:.2f}%, "
                    f"min={value['min']:.4f}, "
                    f"max={value['max']:.4f}"
                )

        return results

## Const

In [148]:
import ast

path = "data_root/cache/imagenet1000_clsidx_to_labels.txt"

with open(path, "r") as f:
    clsidx_to_labels = ast.literal_eval(f.read())

imagenet_classes = [
    label.split(",")[0].strip()
    for idx, label in sorted(clsidx_to_labels.items())
]
print(len(imagenet_classes))

imagenet_classes_prompts = [f'a photo of {c}' for c in imagenet_classes]




import pandas as pd

path = "data_root/cache/openimage_classes.csv"

openimage_classes = pd.read_csv(
    path,
    header=None,
    names=["class_id", "class_name"],
)["class_name"].tolist()

print(len(openimage_classes))
print(openimage_classes[:10])

openimage_classes_prompts = [f'a photo of {c}' for c in openimage_classes]




coco_classes = [
    "airplane", "apple", "backpack", "banana", "baseball bat",
    "baseball glove", "bear", "bed", "bench", "bicycle", "bird",
    "boat", "book", "bottle", "bowl", "broccoli", "bus", "cake",
    "car", "carrot", "cat", "cell phone", "chair", "clock",
    "couch", "cow", "cup", "dining table", "dog", "donut",
    "elephant", "fire hydrant", "fork", "frisbee", "giraffe",
    "hair drier", "handbag", "horse", "hot dog", "keyboard",
    "kite", "knife", "laptop", "microwave", "motorcycle", "mouse",
    "orange", "oven", "parking meter", "person", "pizza",
    "potted plant", "refrigerator", "remote", "sandwich", "scissors",
    "sheep", "sink", "skateboard", "skis", "snowboard", "spoon",
    "sports ball", "stop sign", "suitcase", "surfboard",
    "teddy bear", "tennis racket", "tie", "toaster", "toilet",
    "toothbrush", "traffic light", "train", "truck", "tv",
    "umbrella", "vase", "wine glass", "zebra",
]
coco_classes_prompts = [f'a photo of {c}' for c in coco_classes]

print(len(coco_classes_prompts))

1000
601
['Tortoise', 'Container', 'Magpie', 'Sea turtle', 'Football', 'Ambulance', 'Ladder', 'Toothbrush', 'Syringe', 'Sink']
80


In [108]:
main_experiment_target_concepts = [
'Margot Robbie',
 'Barack Obama',
 'David Beckham',
 'Rihanna',
 'a painting in the style of Picasso',
 'a painting in the style of Van Gogh',
 'a painting in the style of Claude Monet',
 'a painting in the style of Jackson Pollock',
 'Mickey Mouse',
 'R2D2',
 'Grumpy Cat',
 'Macbook',
 'naked person']

main_experiment_generic_concepts =[
    'person',
 'person',
 'person',
 'person',
 'a painting in the style of artist',
 'a painting in the style of artist',
 'a painting in the style of artist',
 'a painting in the style of artist',
 'cartoon character',
 'robot',
 'cat',
 'laptop',
 'dressed person']

main_experiment_target_prompts = [
    f"a photo of {c}" if "painting" not in c else c
    for c in main_experiment_target_concepts
]

main_experiment_generic_prompts = [
    f"a photo of {c}" if "painting" not in c else c
    for c in main_experiment_generic_concepts
]


In [142]:
artists_50 = [
'Van Gogh', 'Claude Monet', 'Picasso', 'Jackson Pollock', 'Akira Toriyama',
'George Stubbs', 'Carne Griffiths', 'Alfred Sisley', 'Charles Hermans', 'Pieter Bruegel The Elder',

# 'Aaron Horkey', 'Frank Bowling', 'Brian Mashburn', 'Amedeo Modigliani', 'Edouard Cortes',
# 'Fernand Khnopff', 'Jean Baptiste Simeon Chardin', 'Benozzo Gozzoli', 'Jehan Georges Vibert', 'Henri De Toulouse Lautrec',

'William Turner', 'Alex Garant', 'Pieter De Hooch', 'Rosalyn Drexler', 'William James Glackens',
'Charles Blackman', 'Leroy Neiman', 'Frans Hals', 'Pierre Auguste Renoir', 'Brian Kesinger',

'Jay Defeo', 'N.C. Wyeth', 'Franz Xaver Winterhalter', 'Mary Cassatt', 'Gustav Klimt',
'Edmund Charles Tarbell', 'Paul Delvaux', 'Arthur Hughes', 'Chiharu Shiota', 'Charline Von Heyl',

'Romare Bearden', 'Egon Schiele', 'Thomas Moran', 'Georges Seurat', 'Casey Weldon',
'Hans Hofmann', 'Pietro Perugino', 'Joan Miro', 'Alberto Vargas', 'Edwin Henry Landseer'
]



celebs_50 = ["Bill Clinton", "Jared Leto", "Shirley Temple", "Shia LaBeouf", "Zayn Malik",
         "George Clooney", "Jennifer Lawrence", "Ronald Reagan", "Anjelica Huston", "Charlie Sheen",
         "Meryl Streep", "Angelina Jolie", "Bruce Willis", "Drake", "Demi Lovato",
         "Barack Obama", "Whoopi Goldberg", "Philip Seymour Hoffman", "Margot Robbie", "Nick Jonas",
         "Glenn Close", "Emma Stone", "Olivia Wilde", "Ryan Gosling", "Richard Gere",
         "Tom Hanks", "Jennifer Aniston", "Tom Hiddleston", "Jennifer Lopez", "Justin Timberlake",
         "Drew Barrymore", "Idris Elba", "Kate Winslet", "Adriana Lima", "Jake Gyllenhaal",
         "Frida Kahlo", "Ricky Gervais", "Jessica Biel", "Hugh Jackman", "Charlize Theron",
         "Beth Behrs", "Reese Witherspoon", "Johnny Depp", "Chris Hemsworth", "Mick Jagger",
         "Jason Momoa", "Andrew Garfield", "Jessica Chastain", "Courteney Cox", "Spike Lee"]


celeb50_target_prompts = [f"a photo of {celeb}" for celeb in celebs_50]
celeb50_generic_prompts = ['a photo of person']*len(celebs_50)


artists50_target_prompts = [f"a painting in the style of {artist}" for artist in artists_50]
artists50_generic_prompts = ['a painting in the style of artist']*len(artists_50)


In [110]:
import random

seed = 123456
rng = random.Random(seed)

random_openimage_classes_prompts = rng.choices(
    openimage_classes_prompts,
    k=10000,
)

print(len(random_openimage_classes_prompts))
print(random_openimage_classes_prompts[:10])



10000
['a photo of Window', 'a photo of Caterpillar', 'a photo of Snowman', 'a photo of Tie', 'a photo of Container', 'a photo of Football helmet', 'a photo of Hot dog', 'a photo of Shower', 'a photo of Bow and arrow', 'a photo of Christmas tree']


## Model

In [14]:

MODEL_ID = "CompVis/stable-diffusion-v1-4"

# Load your pipeline
pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    safety_checker=None
).to(device)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe.set_progress_bar_config(disable=True)

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


In [51]:
# from torch.nn.functional import cosine_similarity

# feature_option = "text"

# p_e, _ = build_features(
#     main_experiment_target_prompts,
#     feature_option=feature_option,
#     is_normalize=True,
#     return_eos=True,
# )

# p_g, _ = build_features(
#     main_experiment_generic_prompts,
#     feature_option=feature_option,
#     is_normalize=True,
#     return_eos=True,
# )

# assert p_e.shape == p_g.shape, f"Shape mismatch: p_e={p_e.shape}, p_g={p_g.shape}"

# cos_sim = cosine_similarity(p_e, p_g, dim=-1)
# # shape: [num_pairs]

# # print("cos_sim:", cos_sim)
# # print("shape:", cos_sim.shape)

# print(f"mean: {cos_sim.mean().item():.4f}")
# print(f"std:  {cos_sim.std(unbiased=False).item():.4f}")
# print(f"min:  {cos_sim.min().item():.4f}")
# print(f"max:  {cos_sim.max().item():.4f}")

# Exp

In [122]:
results_same = summarize_target_generic_similarity(
    target_prompts=main_experiment_target_prompts,
    generic_prompts=main_experiment_generic_prompts,
    feature_option="unet",
    distance_aggregate_option="same_pair",
)

num prompts: 13
[['<|startoftext|>', 'a</w>', 'photo</w>', 'of</w>', 'margot</w>', 'robbie</w>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<

In [123]:
results_same = summarize_target_generic_similarity(
    target_prompts=celeb50_target_prompts,
    generic_prompts=celeb50_generic_prompts,
    feature_option="unet",
    distance_aggregate_option="same_pair",
)

num prompts: 50
[['<|startoftext|>', 'a</w>', 'photo</w>', 'of</w>', 'bill</w>', 'clinton</w>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|

In [145]:
results_same = summarize_target_generic_similarity(
    target_prompts=artists50_target_prompts,
    generic_prompts=artists50_generic_prompts,
    feature_option="unet",
    distance_aggregate_option="same_pair",
)

num prompts: 40
[['<|startoftext|>', 'a</w>', 'painting</w>', 'in</w>', 'the</w>', 'style</w>', 'of</w>', 'van</w>', 'gogh</w>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|e

In [125]:
results_same = summarize_target_generic_similarity(
    target_prompts=coco_classes_prompts,
    generic_prompts=['a photo of object'] * len(coco_classes_prompts),
    feature_option="unet",
    distance_aggregate_option="same_pair",
)

num prompts: 80
[['<|startoftext|>', 'a</w>', 'photo</w>', 'of</w>', 'airplane</w>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>

In [149]:
results_same = summarize_target_generic_similarity(
    target_prompts=openimage_classes_prompts,
    generic_prompts=['a photo of object'] * len(openimage_classes_prompts),
    feature_option="unet",
    distance_aggregate_option="same_pair",
)

num prompts: 601
[['<|startoftext|>', 'a</w>', 'photo</w>', 'of</w>', 'tortoise</w>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|

In [127]:
results_same = summarize_target_generic_similarity(
    target_prompts=imagenet_classes_prompts,
    generic_prompts=imagenet_classes_prompts,
    feature_option="unet",
    distance_aggregate_option="all_pair",
    exclude_samepair=True,
    show_negative_pair=True
)

num prompts: 1000
[['<|startoftext|>', 'a</w>', 'photo</w>', 'of</w>', 'ten', 'ch</w>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftex

In [128]:
results_same = summarize_target_generic_similarity(
    target_prompts=openimage_classes_prompts,
    generic_prompts=openimage_classes_prompts,
    feature_option="unet",
    distance_aggregate_option="all_pair",
    exclude_samepair=True,
    show_negative_pair=True
)

num prompts: 601
[['<|startoftext|>', 'a</w>', 'photo</w>', 'of</w>', 'tortoise</w>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|

In [129]:
results_same = summarize_target_generic_similarity(
    target_prompts=coco_classes_prompts,
    generic_prompts=coco_classes_prompts,
    feature_option="unet",
    distance_aggregate_option="all_pair",
    exclude_samepair=True,
    show_negative_pair=True
)

num prompts: 80
[['<|startoftext|>', 'a</w>', 'photo</w>', 'of</w>', 'airplane</w>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>

In [137]:
results_same = summarize_target_generic_similarity(
    target_prompts=main_experiment_target_prompts,
    generic_prompts=random_openimage_classes_prompts[:len(main_experiment_target_prompts)],
    feature_option="unet",
    distance_aggregate_option="same_pair",
)

num prompts: 13
[['<|startoftext|>', 'a</w>', 'photo</w>', 'of</w>', 'margot</w>', 'robbie</w>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<

In [138]:
results_same = summarize_target_generic_similarity(
    target_prompts=celeb50_target_prompts,
    generic_prompts=random_openimage_classes_prompts[:len(celeb50_target_prompts)],
    feature_option="unet",
    distance_aggregate_option="same_pair",
)

num prompts: 50
[['<|startoftext|>', 'a</w>', 'photo</w>', 'of</w>', 'bill</w>', 'clinton</w>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|

In [144]:
results_same = summarize_target_generic_similarity(
    target_prompts=[c.replace('a painting in the style of', 'a photo of') for c in artists50_target_prompts],
    # generic_prompts=[ c.replace('a photo of', 'a painting in the style of') for c in random_openimage_classes_prompts[:len(artists50_target_prompts)] ],
    generic_prompts=random_openimage_classes_prompts[:len(artists50_target_prompts)],
    feature_option="unet",
    distance_aggregate_option="same_pair",
)

num prompts: 40
[['<|startoftext|>', 'a</w>', 'photo</w>', 'of</w>', 'van</w>', 'gogh</w>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endo

In [146]:
results_same = summarize_target_generic_similarity(
    target_prompts=coco_classes_prompts,
    generic_prompts=random_openimage_classes_prompts[:len(coco_classes_prompts)],
    feature_option="unet",
    distance_aggregate_option="same_pair",

)

num prompts: 80
[['<|startoftext|>', 'a</w>', 'photo</w>', 'of</w>', 'airplane</w>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>

In [150]:
results_same = summarize_target_generic_similarity(
    target_prompts=openimage_classes_prompts,
    generic_prompts=random_openimage_classes_prompts[:len(openimage_classes_prompts)],
    feature_option="unet",
    distance_aggregate_option="same_pair",

)

num prompts: 601
[['<|startoftext|>', 'a</w>', 'photo</w>', 'of</w>', 'tortoise</w>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|

# Effectiveness of Our loss

In [154]:
def summarize_feature_displacement(
    target_prompts,
    generic_prompts,
    unet_e,
    unet_0=None,
    is_normalize=True,
    return_eos=True,
    eps=1e-8,
    make_pairwise_df=True,
    verbose=True,
    print_only_summary=True,
):
    """
    Summarize empirical feature displacement induced by an edited UNet.

    Push constraint measurement:
        cos(W_e p_e, W_0 p_e)

    Pull constraint measurement:
        cos(W_e p_e, W_0 p_g) / cos(W_0 p_e, W_0 p_g)

    where:
        W_e = edited UNet
        W_0 = original UNet
        p_e = target / erased concept prompt
        p_g = generic anchor prompt
    """

    global pipe

    if unet_0 is None:
        unet_0 = pipe.unet

    if isinstance(target_prompts, str):
        target_prompts = [target_prompts]

    if isinstance(generic_prompts, str):
        generic_prompts = [generic_prompts]

    assert len(target_prompts) == len(generic_prompts), (
        "target_prompts and generic_prompts must have the same length. "
        f"Got {len(target_prompts)} and {len(generic_prompts)}."
    )

    # ==================================================
    # 1. Build UNet-projected features
    # ==================================================

    Wpe, _ = build_features(
        target_prompts,
        feature_option="unet",
        is_normalize=is_normalize,
        return_eos=return_eos,
        unet=unet_e,
    )

    W0pe, _ = build_features(
        target_prompts,
        feature_option="unet",
        is_normalize=is_normalize,
        return_eos=return_eos,
        unet=unet_0,
    )

    W0pg, _ = build_features(
        generic_prompts,
        feature_option="unet",
        is_normalize=is_normalize,
        return_eos=return_eos,
        unet=unet_0,
    )

    assert set(Wpe.keys()) == set(W0pe.keys()) == set(W0pg.keys()), (
        "Layer keys do not match across Wpe, W0pe, and W0pg."
    )

    layer_names = list(Wpe.keys())

    # ==================================================
    # 2. Compute push and pull measurements
    # ==================================================

    push_by_layer = {}
    pull_num_by_layer = {}
    pull_den_by_layer = {}
    pull_ratio_by_layer = {}

    for layer_name in layer_names:
        edited_target = Wpe[layer_name]      # W_e p_e
        original_target = W0pe[layer_name]  # W_0 p_e
        original_generic = W0pg[layer_name] # W_0 p_g

        assert edited_target.shape == original_target.shape, (
            f"Shape mismatch at {layer_name}: "
            f"Wpe={edited_target.shape}, W0pe={original_target.shape}"
        )

        assert edited_target.shape == original_generic.shape, (
            f"Shape mismatch at {layer_name}: "
            f"Wpe={edited_target.shape}, W0pg={original_generic.shape}"
        )

        push_cos = _paired_cosine(
            edited_target,
            original_target,
        )

        pull_num = _paired_cosine(
            edited_target,
            original_generic,
        )

        pull_den = _paired_cosine(
            original_target,
            original_generic,
        )

        pull_den_safe = torch.where(
            pull_den.abs() < eps,
            torch.full_like(pull_den, eps),
            pull_den,
        )

        pull_ratio = pull_num / pull_den_safe

        push_by_layer[layer_name] = push_cos
        pull_num_by_layer[layer_name] = pull_num
        pull_den_by_layer[layer_name] = pull_den
        pull_ratio_by_layer[layer_name] = pull_ratio

    # Shape: [num_layers, num_pairs]
    push_tensor = torch.stack(
        [push_by_layer[layer_name] for layer_name in layer_names],
        dim=0,
    )

    pull_num_tensor = torch.stack(
        [pull_num_by_layer[layer_name] for layer_name in layer_names],
        dim=0,
    )

    pull_den_tensor = torch.stack(
        [pull_den_by_layer[layer_name] for layer_name in layer_names],
        dim=0,
    )

    pull_ratio_tensor = torch.stack(
        [pull_ratio_by_layer[layer_name] for layer_name in layer_names],
        dim=0,
    )

    # ==================================================
    # 3. Overall statistics
    # ==================================================

    push_all_stats = compute_stats(push_tensor)
    pull_num_all_stats = compute_stats(pull_num_tensor)
    pull_den_all_stats = compute_stats(pull_den_tensor)
    pull_ratio_all_stats = compute_stats(pull_ratio_tensor)

    # ==================================================
    # 4. Layer-wise statistics across concepts
    # ==================================================

    layer_rows = []

    for layer_idx, layer_name in enumerate(layer_names):
        push_stats = compute_stats(push_tensor[layer_idx])
        pull_num_stats = compute_stats(pull_num_tensor[layer_idx])
        pull_den_stats = compute_stats(pull_den_tensor[layer_idx])
        pull_ratio_stats = compute_stats(pull_ratio_tensor[layer_idx])

        layer_rows.append(
            {
                "layer_idx": layer_idx,
                "layer_name": layer_name,

                "push_mean": push_stats["mean"],
                "push_std": push_stats["std"],
                "push_cv_percent": push_stats["cv_percent"],
                "push_min": push_stats["min"],
                "push_max": push_stats["max"],

                "pull_num_mean": pull_num_stats["mean"],
                "pull_num_std": pull_num_stats["std"],
                "pull_num_min": pull_num_stats["min"],
                "pull_num_max": pull_num_stats["max"],

                "pull_den_mean": pull_den_stats["mean"],
                "pull_den_std": pull_den_stats["std"],
                "pull_den_min": pull_den_stats["min"],
                "pull_den_max": pull_den_stats["max"],

                "pull_ratio_mean": pull_ratio_stats["mean"],
                "pull_ratio_std": pull_ratio_stats["std"],
                "pull_ratio_cv_percent": pull_ratio_stats["cv_percent"],
                "pull_ratio_min": pull_ratio_stats["min"],
                "pull_ratio_max": pull_ratio_stats["max"],
            }
        )

    layerwise_df = pd.DataFrame(layer_rows)

    # ==================================================
    # 5. Concept-wise statistics across layers
    # ==================================================

    pair_names = [
        f"{t}  <->  {g}"
        for t, g in zip(target_prompts, generic_prompts)
    ]

    conceptwise_df = None

    if make_pairwise_df:
        concept_rows = []

        for pair_idx, pair_name in enumerate(pair_names):
            push_stats = compute_stats(push_tensor[:, pair_idx])
            pull_num_stats = compute_stats(pull_num_tensor[:, pair_idx])
            pull_den_stats = compute_stats(pull_den_tensor[:, pair_idx])
            pull_ratio_stats = compute_stats(pull_ratio_tensor[:, pair_idx])

            concept_rows.append(
                {
                    "pair_idx": pair_idx,
                    "pair_name": pair_name,
                    "target_prompt": target_prompts[pair_idx],
                    "generic_prompt": generic_prompts[pair_idx],

                    "push_mean": push_stats["mean"],
                    "push_std": push_stats["std"],
                    "push_cv_percent": push_stats["cv_percent"],
                    "push_min": push_stats["min"],
                    "push_max": push_stats["max"],

                    "pull_num_mean": pull_num_stats["mean"],
                    "pull_num_std": pull_num_stats["std"],
                    "pull_num_min": pull_num_stats["min"],
                    "pull_num_max": pull_num_stats["max"],

                    "pull_den_mean": pull_den_stats["mean"],
                    "pull_den_std": pull_den_stats["std"],
                    "pull_den_min": pull_den_stats["min"],
                    "pull_den_max": pull_den_stats["max"],

                    "pull_ratio_mean": pull_ratio_stats["mean"],
                    "pull_ratio_std": pull_ratio_stats["std"],
                    "pull_ratio_cv_percent": pull_ratio_stats["cv_percent"],
                    "pull_ratio_min": pull_ratio_stats["min"],
                    "pull_ratio_max": pull_ratio_stats["max"],
                }
            )

        conceptwise_df = pd.DataFrame(concept_rows)

    # ==================================================
    # 6. Consistency summaries
    # ==================================================

    push_layer_means = push_tensor.mean(dim=1)
    push_concept_means = push_tensor.mean(dim=0)

    pull_layer_means = pull_ratio_tensor.mean(dim=1)
    pull_concept_means = pull_ratio_tensor.mean(dim=0)

    push_consistency_summary = {
        "summary_of_layer_means": compute_stats(push_layer_means),
        "summary_of_concept_means": compute_stats(push_concept_means),
        "summary_of_layer_stds": compute_stats(push_tensor.std(dim=1, unbiased=False)),
        "summary_of_concept_stds": compute_stats(push_tensor.std(dim=0, unbiased=False)),
    }

    pull_consistency_summary = {
        "summary_of_layer_means": compute_stats(pull_layer_means),
        "summary_of_concept_means": compute_stats(pull_concept_means),
        "summary_of_layer_stds": compute_stats(pull_ratio_tensor.std(dim=1, unbiased=False)),
        "summary_of_concept_stds": compute_stats(pull_ratio_tensor.std(dim=0, unbiased=False)),
    }

    # ==================================================
    # 7. Concise summary
    # ==================================================

    concise_summary = {
        "push measurement": {
            "comment": "cos(W_e p_e, W_0 p_e), pooled over all cross-attention K/V layers and concepts.",
            "mean": push_all_stats["mean"],
            "std": push_all_stats["std"],
            "cv_percent": push_all_stats["cv_percent"],
            "min": push_all_stats["min"],
            "max": push_all_stats["max"],
        },

        "pull numerator": {
            "comment": "cos(W_e p_e, W_0 p_g), pooled over all cross-attention K/V layers and target--generic pairs.",
            "mean": pull_num_all_stats["mean"],
            "std": pull_num_all_stats["std"],
            "cv_percent": pull_num_all_stats["cv_percent"],
            "min": pull_num_all_stats["min"],
            "max": pull_num_all_stats["max"],
        },

        "pull denominator": {
            "comment": "cos(W_0 p_e, W_0 p_g), used to normalize the relative pull ratio.",
            "mean": pull_den_all_stats["mean"],
            "std": pull_den_all_stats["std"],
            "cv_percent": pull_den_all_stats["cv_percent"],
            "min": pull_den_all_stats["min"],
            "max": pull_den_all_stats["max"],
        },

        "relative pull ratio": {
            "comment": "cos(W_e p_e, W_0 p_g) / cos(W_0 p_e, W_0 p_g), pooled over all layers and pairs.",
            "mean": pull_ratio_all_stats["mean"],
            "std": pull_ratio_all_stats["std"],
            "cv_percent": pull_ratio_all_stats["cv_percent"],
            "min": pull_ratio_all_stats["min"],
            "max": pull_ratio_all_stats["max"],
        },

        "push layer-wise consistency": {
            "comment": "Variation across layers after averaging push values over concepts.",
            "mean": push_consistency_summary["summary_of_layer_means"]["mean"],
            "std": push_consistency_summary["summary_of_layer_means"]["std"],
            "cv_percent": push_consistency_summary["summary_of_layer_means"]["cv_percent"],
            "min": push_consistency_summary["summary_of_layer_means"]["min"],
            "max": push_consistency_summary["summary_of_layer_means"]["max"],
        },

        "push concept-wise consistency": {
            "comment": "Variation across concepts after averaging push values over layers.",
            "mean": push_consistency_summary["summary_of_concept_means"]["mean"],
            "std": push_consistency_summary["summary_of_concept_means"]["std"],
            "cv_percent": push_consistency_summary["summary_of_concept_means"]["cv_percent"],
            "min": push_consistency_summary["summary_of_concept_means"]["min"],
            "max": push_consistency_summary["summary_of_concept_means"]["max"],
        },

        "pull layer-wise consistency": {
            "comment": "Variation across layers after averaging relative pull ratios over concepts.",
            "mean": pull_consistency_summary["summary_of_layer_means"]["mean"],
            "std": pull_consistency_summary["summary_of_layer_means"]["std"],
            "cv_percent": pull_consistency_summary["summary_of_layer_means"]["cv_percent"],
            "min": pull_consistency_summary["summary_of_layer_means"]["min"],
            "max": pull_consistency_summary["summary_of_layer_means"]["max"],
        },

        "pull concept-wise consistency": {
            "comment": "Variation across concepts after averaging relative pull ratios over layers.",
            "mean": pull_consistency_summary["summary_of_concept_means"]["mean"],
            "std": pull_consistency_summary["summary_of_concept_means"]["std"],
            "cv_percent": pull_consistency_summary["summary_of_concept_means"]["cv_percent"],
            "min": pull_consistency_summary["summary_of_concept_means"]["min"],
            "max": pull_consistency_summary["summary_of_concept_means"]["max"],
        },
    }

    # ==================================================
    # 8. Collect results
    # ==================================================

    results = {
        "push_tensor": push_tensor,
        "pull_num_tensor": pull_num_tensor,
        "pull_den_tensor": pull_den_tensor,
        "pull_ratio_tensor": pull_ratio_tensor,

        "push_by_layer": push_by_layer,
        "pull_num_by_layer": pull_num_by_layer,
        "pull_den_by_layer": pull_den_by_layer,
        "pull_ratio_by_layer": pull_ratio_by_layer,

        "layer_names": layer_names,
        "pair_names": pair_names,
        "target_prompts": target_prompts,
        "generic_prompts": generic_prompts,

        "push_all_stats": push_all_stats,
        "pull_num_all_stats": pull_num_all_stats,
        "pull_den_all_stats": pull_den_all_stats,
        "pull_ratio_all_stats": pull_ratio_all_stats,

        "layerwise_df": layerwise_df,
        "conceptwise_df": conceptwise_df,

        "push_consistency_summary": push_consistency_summary,
        "pull_consistency_summary": pull_consistency_summary,

        "concise_summary": concise_summary,
    }

    # ==================================================
    # 9. Print report
    # ==================================================

    if verbose:
        if print_only_summary:
            for key, value in concise_summary.items():
                print(f"\n{key}")
                print("-" * len(key))
                print(f"comment: {value['comment']}")
                print(
                    f"mean={value['mean']:.4f}, "
                    f"std={value['std']:.4f}, "
                    f"CV={value['cv_percent']:.2f}%, "
                    f"min={value['min']:.4f}, "
                    f"max={value['max']:.4f}"
                )

        else:
            print("\n==============================")
            print("Feature displacement tensor shapes")
            print("==============================")
            print(f"push_tensor:       {push_tensor.shape}")
            print(f"pull_num_tensor:   {pull_num_tensor.shape}")
            print(f"pull_den_tensor:   {pull_den_tensor.shape}")
            print(f"pull_ratio_tensor: {pull_ratio_tensor.shape}")
            print(f"num_layers = {len(layer_names)}")
            print(f"num_pairs = {len(target_prompts)}")

            print("\n==============================")
            print("Overall pooled stats")
            print("==============================")
            print("push:", push_all_stats)
            print("pull numerator:", pull_num_all_stats)
            print("pull denominator:", pull_den_all_stats)
            print("relative pull ratio:", pull_ratio_all_stats)

            print("\n==============================")
            print("Layer-wise statistics")
            print("==============================")
            _display_or_print(layerwise_df)

            if make_pairwise_df:
                print("\n==============================")
                print("Concept-wise statistics")
                print("==============================")
                _display_or_print(conceptwise_df)

            print("\n==============================")
            print("Consistency summaries")
            print("==============================")
            print("push:", push_consistency_summary)
            print("pull:", pull_consistency_summary)

    return results

In [244]:


unet_e = deepcopy(pipe.unet)


# load_unet_weight_path = "data_root/logs/esd/study/esd-x-kv.bG.fG.T999-1000.peUG-PS1.00_1.00AtE0.20Ir0.80P32.00-N0.00G0.00-mce-1.rs-rg_U.mrobbie_sd1.4.bf16.bs4_r0/step1000.safetensors"
load_unet_weight_path = "data_root/logs/esd/study/esd-x-kv.bG.fG.T999-1000.peUG-PS1.00_1.00AtE0.40Ir0.80P1.00-N0.00G0.00-mte-1.rs.pC-rg-TEb_U.adriver_sd1.4.bf16.bs4_r0/step1000.safetensors"

# load_unet_weight_path = "data_root/logs/esd/study/esd-x-kv.bG.fG.T999-1000.peUG-PS1.00_1.00AtE0.20I0.40P1.00-N0.00G0.00-mt.rs.pc-rg-TEb_U.adriver_sd1.4.bf16.bs4_r0/step1000.safetensors"
# load_unet_weight_path = "data_root/logs/esd/study/esd-x-kv.bG.fG.T999-1000.peUG-PS1.00_1.00AtE0.40I0.40P1.00-N0.00G0.00-mte-1.rs.pCE-rg-TEb_U.adriver_sd1.4.bf16.bs4_r0/step1000.safetensors"


# load_unet_weight_path = "data_root/logs/esd/study/esd-x-kv.bG.fG.T999-1000.peUG-PS1.00_1.00AtL2E750.00I500.00P1.00-N0.00G0.00-mte0.rs_U.adriver_sd1.4.bf16.bs4_r0/step1000.safetensors"


# esd-x-kv.bG.fG.T999-1000.peUG-PS1.00_1.00AtL2E750.00I0.00-0.00P1.00-N0.00G0.00-mte0.rs_U.adriver_sd1.4.bf16.bs4_r0/

# esd-x-kv.bG.fG.T999-1000.peUG-PS1.00_1.00AtE0.40I0.40P1.00-N0.00G0.00-mte-1.rs.pCE-rg-TEb_U.adriver_sd1.4.bf16.bs4_r0/


unet_e.load_state_dict(load_file(load_unet_weight_path), strict=False)





_IncompatibleKeys(missing_keys=['conv_in.weight', 'conv_in.bias', 'time_embedding.linear_1.weight', 'time_embedding.linear_1.bias', 'time_embedding.linear_2.weight', 'time_embedding.linear_2.bias', 'down_blocks.0.attentions.0.norm.weight', 'down_blocks.0.attentions.0.norm.bias', 'down_blocks.0.attentions.0.proj_in.weight', 'down_blocks.0.attentions.0.proj_in.bias', 'down_blocks.0.attentions.0.transformer_blocks.0.norm1.weight', 'down_blocks.0.attentions.0.transformer_blocks.0.norm1.bias', 'down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_q.weight', 'down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_k.weight', 'down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_v.weight', 'down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_out.0.weight', 'down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_out.0.bias', 'down_blocks.0.attentions.0.transformer_blocks.0.norm2.weight', 'down_blocks.0.attentions.0.transformer_blocks.0.norm2.bias', 'down_blocks.0.attentions.0.t

In [245]:
results = summarize_feature_displacement(
    target_prompts=['Adam Driver'],
    generic_prompts=['the person'],


    # target_prompts=['a photo of Margot Robbie'],
    # generic_prompts=['a photo of the person'],

    unet_e=unet_e,
    unet_0=pipe.unet,
    is_normalize=True,
    return_eos=True,
    verbose=True,
    print_only_summary=True,
)

num prompts: 1
[['<|startoftext|>', 'adam</w>', 'driver</w>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|

In [ ]:
comment: cos(W_e p_e, W_0 p_g) / cos(W_0 p_e, W_0 p_g), pooled over all layers and pairs.
# ours
mean=0.8294, std=0.0899, CV=10.84%, min=0.7036, max=1.0625
mean=0.6893, std=0.0460, CV=6.68%, min=0.5537, max=0.7642


# absolute push
mean=1.6022, std=0.7652, CV=47.76%, min=0.6865, max=3.5234

# L2
mean=2.3848, std=1.2020, CV=50.41%, min=0.6704, max=5.4141



In [252]:
import torch
import pandas as pd
from torch.nn.functional import cosine_similarity

# --------------------------------------------------
# Build UNet features
# --------------------------------------------------

feature_option = "unet"

p_e, _ = build_features(
    main_experiment_target_prompts,
    feature_option=feature_option,
    is_normalize=True,
    return_eos=True,
)

p_g, _ = build_features(
    main_experiment_generic_prompts,
    feature_option=feature_option,
    is_normalize=True,
    return_eos=True,
)

assert set(p_e.keys()) == set(p_g.keys()), "Layer keys do not match"

layer_names = list(p_e.keys())
concept_names = main_experiment_target_prompts

assert len(concept_names) == len(main_experiment_generic_prompts), (
    f"Number of target and generic prompts do not match: "
    f"{len(concept_names)} vs {len(main_experiment_generic_prompts)}"
)


# --------------------------------------------------
# Helper function
# --------------------------------------------------

def compute_stats(x):
    x = x.float().flatten()

    return {
        "mean": x.mean().item(),
        "average": x.mean().item(),
        "std": x.std(unbiased=False).item(),
        "min": x.min().item(),
        "max": x.max().item(),
    }


# --------------------------------------------------
# Compute cosine similarity for each UNet layer
# --------------------------------------------------

cos_sim_by_layer = {}

for layer_name in layer_names:
    assert p_e[layer_name].shape == p_g[layer_name].shape, (
        f"Shape mismatch at {layer_name}: "
        f"p_e={p_e[layer_name].shape}, p_g={p_g[layer_name].shape}"
    )

    cos_sim = cosine_similarity(
        p_e[layer_name],
        p_g[layer_name],
        dim=-1,
    )

    # If the UNet feature has extra dimensions, e.g. spatial/token dimensions,
    # average them to obtain one cosine value per concept pair.
    #
    # Example:
    # [num_concepts, H, W] -> [num_concepts]
    # [num_concepts, num_tokens] -> [num_concepts]
    if cos_sim.ndim > 1:
        cos_sim = cos_sim.mean(dim=tuple(range(1, cos_sim.ndim)))

    assert cos_sim.ndim == 1, (
        f"Unexpected cosine shape at {layer_name}: {cos_sim.shape}"
    )

    cos_sim_by_layer[layer_name] = cos_sim


# --------------------------------------------------
# Stack into matrix: [num_layers, num_concepts]
# --------------------------------------------------

cos_matrix = torch.stack(
    [cos_sim_by_layer[layer_name] for layer_name in layer_names],
    dim=0,
)

print("cos_matrix shape:", cos_matrix.shape)
# Expected: [num_layers, num_concepts]


# ==================================================
# 1. ALL STATS
# ==================================================
# Statistics over all layer-concept values.

all_stats = compute_stats(cos_matrix)

print("\n==============================")
print("1. All stats")
print("==============================")
print(all_stats)


# ==================================================
# 2. ACROSS LAYERS
# ==================================================
# For each concept, compute statistics across layers.
# Input shape for each concept: [num_layers]

across_layers_stats = {}

for concept_idx, concept_name in enumerate(concept_names):
    values_across_layers = cos_matrix[:, concept_idx]
    across_layers_stats[concept_name] = compute_stats(values_across_layers)

across_layers_df = pd.DataFrame(across_layers_stats).T

print("\n==============================")
print("2. Across layers: per-concept stats")
print("==============================")
display(across_layers_df)


# --------------------------------------------------
# Summary of across-layers result
# --------------------------------------------------
# First reduce layers, then summarize across concepts.

concept_mean_over_layers = cos_matrix.mean(dim=0)
concept_std_over_layers = cos_matrix.std(dim=0, unbiased=False)

across_layers_summary = {
    "summary_of_concept_means_over_layers": compute_stats(concept_mean_over_layers),
    "summary_of_concept_stds_over_layers": compute_stats(concept_std_over_layers),
}

print("\n==============================")
print("2b. Summary of across-layers stats")
print("==============================")
print(across_layers_summary)


# ==================================================
# 3. ACROSS CONCEPTS
# ==================================================
# For each layer, compute statistics across concepts.
# Input shape for each layer: [num_concepts]

across_concepts_stats = {}

for layer_idx, layer_name in enumerate(layer_names):
    values_across_concepts = cos_matrix[layer_idx, :]
    across_concepts_stats[layer_name] = compute_stats(values_across_concepts)

across_concepts_df = pd.DataFrame(across_concepts_stats).T

print("\n==============================")
print("3. Across concepts: per-layer stats")
print("==============================")
display(across_concepts_df)


# --------------------------------------------------
# Summary of across-concepts result
# --------------------------------------------------
# First reduce concepts, then summarize across layers.

layer_mean_over_concepts = cos_matrix.mean(dim=1)
layer_std_over_concepts = cos_matrix.std(dim=1, unbiased=False)

across_concepts_summary = {
    "summary_of_layer_means_over_concepts": compute_stats(layer_mean_over_concepts),
    "summary_of_layer_stds_over_concepts": compute_stats(layer_std_over_concepts),
}

print("\n==============================")
print("3b. Summary of across-concepts stats")
print("==============================")
print(across_concepts_summary)


# ==================================================
# 4. Compact final report
# ==================================================

print("\n==============================")
print("Final compact report")
print("==============================")

print("\nAll layer-concept values:")
print(all_stats)

print("\nAcross layers summary:")
print(across_layers_summary)

print("\nAcross concepts summary:")
print(across_concepts_summary)

num prompts: 13
[['<|startoftext|>', 'a</w>', 'photo</w>', 'of</w>', 'margot</w>', 'robbie</w>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<

,mean,average,std,min,max
a photo of Margot Robbie,0.442886,0.442886,0.153464,0.273682,0.809082
a photo of Barack Obama,0.600304,0.600304,0.125453,0.397217,0.852051
a photo of David Beckham,0.468517,0.468517,0.151706,0.247437,0.786621
a photo of Rihanna,0.549019,0.549019,0.133865,0.377930,0.825684
a painting in the style of Picasso,0.856659,0.856659,0.056684,0.757812,0.941895
a painting in the style of Van Gogh,0.812698,0.812698,0.065806,0.701172,0.924316
a painting in the style of Claude Monet,0.713486,0.713486,0.094770,0.543457,0.861328
a painting in the style of Jackson Pollock,0.638718,0.638718,0.111497,0.448242,0.848633
a photo of Mickey Mouse,0.681396,0.681396,0.084381,0.562500,0.835938
a photo of R2D2,0.617836,0.617836,0.123099,0.480957,0.850098



2b. Summary of across-layers stats
{'summary_of_concept_means_over_layers': {'mean': 0.6584097146987915, 'average': 0.6584097146987915, 'std': 0.12615551054477692, 'min': 0.44287109375, 'max': 0.8564453125}, 'summary_of_concept_stds_over_layers': {'mean': 0.10291232168674469, 'average': 0.10291232168674469, 'std': 0.032887835055589676, 'min': 0.04736328125, 'max': 0.1534423828125}}

3. Across concepts: per-layer stats


,mean,average,std,min,max
down_blocks.0.attentions.0.transformer_blocks.0.attn2.to_k.weight,0.656457,0.656457,0.159551,0.358887,0.877930
down_blocks.0.attentions.0.transformer_blocks.0.attn2.to_v.weight,0.739070,0.739070,0.113928,0.524414,0.902832
down_blocks.0.attentions.1.transformer_blocks.0.attn2.to_k.weight,0.693923,0.693923,0.151372,0.413086,0.921875
down_blocks.0.attentions.1.transformer_blocks.0.attn2.to_v.weight,0.828050,0.828050,0.062376,0.721191,0.926758
down_blocks.1.attentions.0.transformer_blocks.0.attn2.to_k.weight,0.651067,0.651067,0.151680,0.378906,0.871094
down_blocks.1.attentions.0.transformer_blocks.0.attn2.to_v.weight,0.743840,0.743840,0.108017,0.552246,0.905273
down_blocks.1.attentions.1.transformer_blocks.0.attn2.to_k.weight,0.666016,0.666016,0.153193,0.358887,0.890137
down_blocks.1.attentions.1.transformer_blocks.0.attn2.to_v.weight,0.825796,0.825796,0.059394,0.723145,0.916504
down_blocks.2.attentions.0.transformer_blocks.0.attn2.to_k.weight,0.631235,0.631235,0.148221,0.358643,0.853516
down_blocks.2.attentions.0.transformer_blocks.0.attn2.to_v.weight,0.566237,0.566237,0.154990,0.312012,0.810547



3b. Summary of across-concepts stats
{'summary_of_layer_means_over_concepts': {'mean': 0.65850830078125, 'average': 0.65850830078125, 'std': 0.09744780510663986, 'min': 0.54150390625, 'max': 0.8544921875}, 'summary_of_layer_stds_over_concepts': {'mean': 0.12944793701171875, 'average': 0.12944793701171875, 'std': 0.036515962332487106, 'min': 0.04693603515625, 'max': 0.1978759765625}}

Final compact report

All layer-concept values:
{'mean': 0.6584863066673279, 'average': 0.6584863066673279, 'std': 0.16610689461231232, 'min': 0.2474365234375, 'max': 0.94189453125}

Across layers summary:
{'summary_of_concept_means_over_layers': {'mean': 0.6584097146987915, 'average': 0.6584097146987915, 'std': 0.12615551054477692, 'min': 0.44287109375, 'max': 0.8564453125}, 'summary_of_concept_stds_over_layers': {'mean': 0.10291232168674469, 'average': 0.10291232168674469, 'std': 0.032887835055589676, 'min': 0.04736328125, 'max': 0.1534423828125}}

Across concepts summary:
{'summary_of_layer_means_over_

In [ ]:
layer_wise_stats.keys()

dict_keys(['mid_block.attentions.0.transformer_blocks.0.attn2.to_k.weight'])

1
1
1
1
1
1
1
1
1
1
1
1
1


In [207]:
features, tokens = build_features(prompts, feature_option=feature_option, is_normalize=True)

num prompts: 13


In [208]:
features.shape

torch.Size([13, 77, 768])

In [205]:
len(tokens)

77

In [195]:
prompts

['a photo of Margot Robbie',
 'a photo of Barack Obama',
 'a photo of David Beckham',
 'a photo of Rihanna',
 'a painting in the style of Picasso',
 'a painting in the style of Van Gogh',
 'a painting in the style of Claude Monet',
 'a painting in the style of Jackson Pollock',
 'a photo of Mickey Mouse',
 'a photo of R2D2',
 'a photo of Grumpy Cat',
 'a photo of Macbook',
 'a photo of naked person']

In [184]:
features

tensor([[-0.0089,  0.0005, -0.0012,  ..., -0.0112, -0.0070,  0.0015],
        [ 0.0010, -0.0464,  0.0108,  ..., -0.0184,  0.0342,  0.0233],
        [ 0.0414,  0.0048,  0.0282,  ..., -0.0754, -0.0412, -0.0119],
        ...,
        [ 0.0008, -0.0290, -0.0102,  ...,  0.0144, -0.0114, -0.0163],
        [ 0.0009, -0.0291, -0.0100,  ...,  0.0145, -0.0118, -0.0160],
        [-0.0025, -0.0275, -0.0095,  ...,  0.0128, -0.0116, -0.0180]],
       device='cuda:2', dtype=torch.float16, grad_fn=<DivBackward0>)

In [175]:
# base_dir = "similariy_analysis/prelim_sim-TEp-E/"
base_dir = "similariy_analysis/prelim_sim-TEp-E/"




all_similarity = []
for concept in os.listdir(base_dir):
# for concept in ['Margot Robbie','Barack Obama','Rihanna','David Beckham']:
    # if 'naked' in concept.lower():
    #     continue
    t = torch.load(f"{base_dir}/{concept}")
    # t = torch.load(os.path.join(base_dir, f"{concept}.safetensors"))
    new_sim = flatten_list([ e.flatten().tolist() for e in t])
    # print(f'{concept}: {new_sim.mean()}')
    for s in new_sim:
        

        if s < 0:
            print(concept)
            print(new_sim)
            continue

    all_similarity += new_sim

all_similarity = torch.tensor(all_similarity)

/tmp/ipykernel_805415/3973065788.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  t = torch.load(f"{base_dir}/{concept}")


In [124]:
print(f'mean: {all_similarity.mean()}')
print(f'max: {all_similarity.max()}')
print(f'min: {all_similarity.min()}')
print(f'std: {all_similarity.std()}')
print(f'standard error: {all_similarity.std() / torch.sqrt(torch.tensor(all_similarity.numel(), dtype=all_similarity.dtype))}')


mean: 0.5395050048828125
max: 0.875
min: 0.3125
std: 0.14870218932628632
standard error: 0.013143541291356087


In [143]:
pretrained_model_name_or_path = 'CompVis/stable-diffusion-v1-4'
unet_0 = UNet2DConditionModel.from_pretrained(
    pretrained_model_name_or_path, subfolder="unet", revision=None,
)


unet_e =  UNet2DConditionModel.from_pretrained(
    pretrained_model_name_or_path, subfolder="unet", revision=None,
)

load_unet_weight_path = "data_root/logs/esd/study/esd-x-kv.bG.fG.T999-1000.peUG-PS1.00_1.00AtE0.40Ir0.40P32.00-N0.00G0.00-mt.rs.pc-rg_U.obama_sd1.4.bf16.bs4_r0/step1000.safetensors"
unet_e.load_state_dict(load_file(load_unet_weight_path), strict=False)


_IncompatibleKeys(missing_keys=['conv_in.weight', 'conv_in.bias', 'time_embedding.linear_1.weight', 'time_embedding.linear_1.bias', 'time_embedding.linear_2.weight', 'time_embedding.linear_2.bias', 'down_blocks.0.attentions.0.norm.weight', 'down_blocks.0.attentions.0.norm.bias', 'down_blocks.0.attentions.0.proj_in.weight', 'down_blocks.0.attentions.0.proj_in.bias', 'down_blocks.0.attentions.0.transformer_blocks.0.norm1.weight', 'down_blocks.0.attentions.0.transformer_blocks.0.norm1.bias', 'down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_q.weight', 'down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_k.weight', 'down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_v.weight', 'down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_out.0.weight', 'down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_out.0.bias', 'down_blocks.0.attentions.0.transformer_blocks.0.norm2.weight', 'down_blocks.0.attentions.0.transformer_blocks.0.norm2.bias', 'down_blocks.0.attentions.0.t

In [144]:
params_e = dict(unet_e.named_parameters())
params_0 = dict(unet_0.named_parameters())

S_e_list = []
S_0_list = []

for name, W_e in params_e.items():
    if not (name.endswith("to_k.weight") or name.endswith("to_v.weight")):
        continue
    if name not in params_0:
        continue

    W_0 = params_0[name].detach()

    _, S_e, _ = torch.linalg.svd(W_e.detach().float(), full_matrices=False)
    _, S_0, _ = torch.linalg.svd(W_0.float(), full_matrices=False)

    S_e_list.append(S_e)
    S_0_list.append(S_0)

In [145]:
for l, S in enumerate(S_0_list):
    print(f"Layer {l}: {S.max()}, {S.min()}")

Layer 0: 5.749772548675537, 0.000743604323361069
Layer 1: 2.966752290725708, 0.0015552969416603446
Layer 2: 5.555182933807373, 0.14330865442752838
Layer 3: 1.8568165302276611, 0.06948772817850113
Layer 4: 6.48639440536499, 0.0020536005031317472
Layer 5: 3.0473856925964355, 0.0020839080680161715
Layer 6: 7.6447930335998535, 0.1391584724187851
Layer 7: 2.2733821868896484, 0.06508976966142654
Layer 8: 7.664798736572266, 0.0011312984861433506
Layer 9: 3.7381837368011475, 6.748215582774719e-06
Layer 10: 6.044878959655762, 0.061044469475746155
Layer 11: 2.4602227210998535, 0.03655282035470009
Layer 12: 7.34505558013916, 0.0005721576744690537
Layer 13: 3.471123456954956, 0.0007574446499347687
Layer 14: 6.798998832702637, 0.04764113947749138
Layer 15: 2.602208137512207, 0.031904127448797226
Layer 16: 10.31320571899414, 0.0006348633323796093
Layer 17: 4.4643096923828125, 0.00019204846466891468
Layer 18: 7.292736530303955, 0.17044559121131897
Layer 19: 2.5266683101654053, 0.007290155626833439
La

In [146]:
for l, S in enumerate(S_e_list):
    print(f"Layer {l}: {S.max()}, {S.min()}")

Layer 0: 5.749772548675537, 0.000743604323361069
Layer 1: 2.966752290725708, 0.0015552969416603446
Layer 2: 5.498087406158447, 0.1560554802417755
Layer 3: 1.8534208536148071, 0.08097627758979797
Layer 4: 6.48639440536499, 0.0020536005031317472
Layer 5: 3.0473856925964355, 0.0020839080680161715
Layer 6: 7.6035871505737305, 0.14741292595863342
Layer 7: 2.2650139331817627, 0.07631860673427582
Layer 8: 7.664798736572266, 0.0011312984861433506
Layer 9: 3.7381837368011475, 6.748215582774719e-06
Layer 10: 5.9835100173950195, 0.06923961639404297
Layer 11: 2.4595558643341064, 0.03694133833050728
Layer 12: 7.34505558013916, 0.0005721576744690537
Layer 13: 3.471123456954956, 0.0007574446499347687
Layer 14: 6.717701435089111, 0.054142165929079056
Layer 15: 2.597247362136841, 0.032197996973991394
Layer 16: 10.31320571899414, 0.0006348633323796093
Layer 17: 4.4643096923828125, 0.00019204846466891468
Layer 18: 7.237303733825684, 0.17964781820774078
Layer 19: 2.524758815765381, 0.008129922673106194
La

In [ ]:
device = 'cuda:2'
pipe = StableDiffusionPipeline.from_pretrained(pretrained_model_name_or_path, unet=unet_0, torch_dtype='fp16', use_safetensors=True).to(device)



Passed `torch_dtype` torch.float32 is not a `torch.dtype`. Defaulting to `torch.float32`.


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

In [ ]:
p_e_prompts = ['a photo of Barack Obama','a photo of Margot Robbie']
p_g_prompts = ['a photo of person']*len(p_e_prompts)
    
    
target_prompt_end_idx =  [ pipe.tokenizer(p )['input_ids'].index(pipe.tokenizer.eos_token_id) for p in p_e_prompts]

with torch.no_grad():
    p_e, _ = pipe.encode_prompt(prompt=p_e_prompts, device=device, num_images_per_prompt=1, do_classifier_free_guidance=True, negative_prompt=len(p_e_prompts)*[''])

p_e_eos = [p_e[i, target_prompt_end_idx[i]] for i in range(len(p_e_prompts))]

In [181]:
all_concepts = ['Margot Robbie',
 'Barack Obama',
 'David Beckham',
 'Rihanna',
 'a painting in the style of Picasso',
 'a painting in the style of Van Gogh',
 'a painting in the style of Claude Monet',
 'a painting in the style of Jackson Pollock',
 'Mickey Mouse',
 'R2D2',
 'Grumpy Cat',
 'Macbook',
 'naked person']

generic_concepts =['person',
 'person',
 'person',
 'person',
 'a painting in the style of artist',
 'a painting in the style of artist',
 'a painting in the style of artist',
 'a painting in the style of artist',
 'cartoon',
 'robot',
 'cat',
 'laptop',
 'dressed person']

In [171]:
import torch
import torch.nn.functional as F

p_e_prompts = [
    f"a photo of {c}" if "painting" not in c else c
    for c in all_concepts
]

p_g_prompts = [
    f"a photo of {c}" if "painting" not in c else c
    for c in generic_concepts
]

def get_eos_indices(prompts, tokenizer, device):
    """
    Return the EOS index for each prompt using the attention mask.
    This is safer than .index(eos_token_id) because CLIP tokenizers often use EOS as padding.
    """
    tokenized = tokenizer(
        prompts,
        padding="max_length",
        max_length=tokenizer.model_max_length,
        truncation=True,
        return_tensors="pt",
    ).to(device)

    eos_indices = tokenized.attention_mask.sum(dim=1) - 1
    return eos_indices


with torch.no_grad():
    # Encode erased prompts
    p_e_embeds, _ = pipe.encode_prompt(
        prompt=p_e_prompts,
        device=device,
        num_images_per_prompt=1,
        do_classifier_free_guidance=True,
        negative_prompt=[""] * len(p_e_prompts),
    )

    # Encode generic prompts
    p_g_embeds, _ = pipe.encode_prompt(
        prompt=p_g_prompts,
        device=device,
        num_images_per_prompt=1,
        do_classifier_free_guidance=True,
        negative_prompt=[""] * len(p_g_prompts),
    )

    # Get EOS token indices
    p_e_eos_idx = get_eos_indices(p_e_prompts, pipe.tokenizer, device)
    p_g_eos_idx = get_eos_indices(p_g_prompts, pipe.tokenizer, device)

    batch_idx = torch.arange(len(p_e_prompts), device=device)

    # Extract EOS embeddings
    p_e_eos = p_e_embeds[batch_idx, p_e_eos_idx]
    p_g_eos = p_g_embeds[batch_idx, p_g_eos_idx]

    # Paired cosine similarity
    eos_cos_sim = F.cosine_similarity(p_e_eos, p_g_eos, dim=-1)

print(eos_cos_sim)

tensor([0.3770, 0.5874, 0.4419, 0.5309, 0.7986, 0.7165, 0.5993, 0.5184, 0.4512,
        0.5560, 0.6192, 0.7951, 0.6279], device='cuda:2')


In [173]:
print(f'mean: {eos_cos_sim.mean()}')
print(f'max: {eos_cos_sim.max()}')
print(f'min: {eos_cos_sim.min()}')
print(f'std: {eos_cos_sim.std()}')


mean: 0.5861077308654785
max: 0.798614501953125
min: 0.3769854009151459
std: 0.12914851307868958
